# Body Performance — Regression Notebook
## Predicting Broad Jump Distance (cm)

**Dataset:** bodyPerformance.csv | **Task:** Regression | **Target:** `broad jump_cm`

---
| Section | Topic |
|---------|-------|
| 1.1 | Dataset Overview |
| 1.2 | Column Understanding |
| 1.3 | Data Type Verification |
| 1.4 | Missing Values Analysis |
| 1.5 | Duplicate Detection |
| 1.6 | Data Validity Checks |
| 1.7 | Univariate Analysis |
| 1.8 | Distribution Analysis |
| 1.9 | Outlier Detection |
| 1.10 | Correlation Analysis |
| 1.11 | Final EDA Summary |
| 2 | Feature Engineering |
| 3 | Model Building & Evaluation |


## Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid', palette='husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
RANDOM_STATE = 42
print("All libraries loaded successfully.")

---
## 1.1 Dataset Overview

First look at the dataset: size, columns, and sample rows.

In [ ]:
df_raw = pd.read_csv('bodyPerformance.csv')
df     = df_raw.copy()

n_rows, n_cols = df.shape
print(f"Number of Rows    : {n_rows:,}")
print(f"Number of Columns : {n_cols}")
print()
print("Column Names:")
for idx, col in enumerate(df.columns, 1):
    print(f"  {idx:>2}. {col}")

In [ ]:
print("Sample rows (first 5):")
display(df.head(5))
print("\nRandom sample (5 rows):")
display(df.sample(5, random_state=42))

### Dataset Description

The **Body Performance Dataset** contains physical fitness measurements from **13,393 individuals** across **12 columns**.
Each row records one person's demographic data (age, gender), body composition (height, weight, body fat),
cardiovascular readings (blood pressure), and four standardised fitness tests.
For this regression notebook, the **target variable is `broad jump_cm`** — the distance achieved in a
standing broad (long) jump test, measured in centimetres. This is a continuous outcome representing explosive
lower-body power and overall athletic performance. The target ranges from 0 to 303 cm with a mean of ~190 cm.
Predicting this value from other physical measurements allows us to understand which bodily attributes
most strongly determine explosive power.


---
## 1.2 Column Understanding

In [ ]:
col_info = pd.DataFrame({
    'Column': ['age','gender','height_cm','weight_kg','body fat_%',
               'diastolic','systolic','gripForce',
               'sit and bend forward_cm','sit-ups counts','broad jump_cm','class'],
    'Meaning': [
        'Age of participant in years',
        'Biological sex: Male (M) or Female (F)',
        'Height in centimetres',
        'Body weight in kilograms',
        'Percentage of body mass composed of fat tissue',
        'Diastolic blood pressure (mmHg) — heart at rest',
        'Systolic blood pressure (mmHg) — heart contracting',
        'Hand grip strength by dynamometer (kg)',
        'Sit-and-reach flexibility test result (cm)',
        'Number of sit-ups completed in timed test',
        'Standing broad jump distance (cm) — TARGET',
        'Overall fitness grade: A=best, D=lowest'
    ],
    'Expected Type': ['Numeric','Categorical','Numeric','Numeric','Numeric',
                      'Numeric','Numeric','Numeric','Numeric','Numeric',
                      'Numeric (TARGET)','Categorical'],
    'Validation Rules': [
        '21-64 (adult range in dataset)',
        'M or F only',
        '> 100 cm',
        '> 0 kg, typically 30-200',
        '3-60 %',
        '40-130 mmHg',
        '60-200 mmHg',
        '>= 0 kg',
        '-30 to 80 cm',
        '0-80 (test cap)',
        '0-400 cm (target; world record ~343 cm)',
        'A, B, C, or D'
    ]
})
display(col_info)

---
## 1.3 Data Type Verification

In [ ]:
dtype_df = df.dtypes.rename('Detected Type').to_frame()
dtype_df['Expected Type'] = ['Numeric','Categorical','Numeric','Numeric','Numeric',
                              'Numeric','Numeric','Numeric','Numeric','Numeric','Numeric','Categorical']
dtype_df['Match'] = dtype_df.apply(
    lambda r: 'Correct' if (
        (r['Expected Type']=='Numeric'     and r['Detected Type'] in ['float64','int64']) or
        (r['Expected Type']=='Categorical' and str(r['Detected Type']) in ['object','str','string'])
    ) else 'MISMATCH', axis=1)
display(dtype_df)

print()
print("Findings:")
print("  - All numeric columns are float64 — correct for mathematical operations.")
print("  - 'age' and 'sit-ups counts' are float64 but represent integer values.")
print("    This is not a critical issue; calculations will still be valid.")
print("  - 'gender' and 'class' are stored as object — correct for categorical data.")
print("  - 'broad jump_cm' (our regression target) is correctly float64.")
print("  - No type mismatches require mandatory correction.")

---
## 1.4 Missing Values Analysis

In [ ]:
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(4)

def assign_strategy(col, pct, series):
    if pct == 0:
        return 'None needed'
    elif pct < 5:
        return 'Drop rows (< 5% impact)'
    elif pct < 30:
        if series.dtype == 'object':
            return 'Fill with Mode'
        elif abs(series.skew()) > 1:
            return 'Fill with Median (skewed)'
        else:
            return 'Fill with Mean (normal)'
    else:
        return 'Drop column (> 30% missing)'

missing_df = pd.DataFrame({'Missing Count': null_counts, 'Missing %': null_pct})
missing_df['Strategy'] = [assign_strategy(c, null_pct[c], df[c]) for c in df.columns]
display(missing_df)

print()
if null_counts.sum() == 0:
    print("RESULT: No missing values in any column. No imputation needed.")
else:
    for col in missing_df.index:
        strat = missing_df.loc[col, 'Strategy']
        if 'Drop rows' in strat:
            df.dropna(subset=[col], inplace=True)
        elif 'Median' in strat:
            df[col].fillna(df[col].median(), inplace=True)
        elif 'Mean' in strat:
            df[col].fillna(df[col].mean(), inplace=True)
        elif 'Mode' in strat:
            df[col].fillna(df[col].mode()[0], inplace=True)
        elif 'Drop column' in strat:
            df.drop(columns=[col], inplace=True)
    print(f"Shape after handling: {df.shape}")

---
## 1.5 Duplicate Detection

In [ ]:
n_dupes = df.duplicated().sum()
print(f"Exact duplicate rows found: {n_dupes}")

if n_dupes > 0:
    print("Duplicate rows:")
    display(df[df.duplicated(keep=False)].sort_values(list(df.columns)))
    df.drop_duplicates(inplace=True)
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicates found. No action required.")

---
## 1.6 Data Validity Checks

Detect values that are logically or physiologically impossible.

In [ ]:
print("=== Validity Check Report ===")
print()

checks = [
    ('systolic',                df['systolic'] <= 40,
     'Systolic BP <= 40 mmHg (impossible for living person)'),
    ('diastolic',               df['diastolic'] <= 40,
     'Diastolic BP <= 40 mmHg (impossible for living person)'),
    ('broad jump_cm',           df['broad jump_cm'] == 0,
     'Broad jump = 0 cm (no attempt or measurement error)'),
    ('broad jump_cm',           df['broad jump_cm'] > 400,
     'Broad jump > 400 cm (exceeds world record of 343 cm)'),
    ('body fat_%',              df['body fat_%'] > 70,
     'Body fat > 70% (extreme/impossible)'),
    ('sit and bend forward_cm', df['sit and bend forward_cm'] > 100,
     'Sit-and-reach > 100 cm (beyond anatomical limit)'),
    ('gripForce',               df['gripForce'] < 0,
     'Negative grip force (impossible)'),
    ('gender',                  ~df['gender'].isin(['M','F']),
     "Invalid gender label"),
    ('class',                   ~df['class'].isin(['A','B','C','D']),
     "Invalid class label"),
]

for col, mask, description in checks:
    count = int(mask.sum())
    tag = f"VIOLATION ({count} rows)" if count > 0 else "OK"
    print(f"  [{tag:<22}]  {description}")

print()
print("Action Plan:")
print("  1. REMOVE rows with BP <= 40 (physiologically dead / data entry error).")
print("  2. CAP sit-and-reach extreme values at anatomical limits.")
print("  3. RETAIN broad jump = 0 (genuine test failures; informative for regression).")
print("  4. No impossible broad jump values > 400 found.")

In [ ]:
before = len(df)
df = df[(df['systolic'] > 40) & (df['diastolic'] > 40)]
print(f"Rows removed (BP <= 40): {before - len(df)}")

old_max = df['sit and bend forward_cm'].max()
df['sit and bend forward_cm'] = df['sit and bend forward_cm'].clip(lower=-25, upper=80)
print(f"sit-and-reach capped — old max was {old_max:.1f}, now capped at 80")
print(f"Dataset shape after fixes: {df.shape}")

---
## 1.7 Univariate Analysis

Descriptive statistics for all numeric columns including the regression target.

In [ ]:
NUM_COLS = ['age','height_cm','weight_kg','body fat_%','diastolic',
            'systolic','gripForce','sit and bend forward_cm',
            'sit-ups counts','broad jump_cm']

stats_table = df[NUM_COLS].agg(['mean','median','std','min','max']).T
stats_table.columns = ['Mean','Median','Std Dev','Min','Max']
stats_table['Skewness'] = df[NUM_COLS].skew().round(3)
stats_table = stats_table.round(3)

print("Descriptive Statistics:")
display(stats_table)

print()
print("=== Target Variable: broad jump_cm ===")
bj = df['broad jump_cm']
print(f"  Mean   : {bj.mean():.2f} cm")
print(f"  Median : {bj.median():.2f} cm")
print(f"  Std Dev: {bj.std():.2f} cm")
print(f"  Min    : {bj.min():.2f} cm")
print(f"  Max    : {bj.max():.2f} cm")
print(f"  Skew   : {bj.skew():.3f} (slight left skew — most participants cluster at moderate performance)")
print()
print("=== Predictor Interpretation Notes ===")
notes = [
    ('age',                     'Mean 36.8yr, right-skewed (0.60). Older participants expected to jump shorter distances.'),
    ('height_cm',               'Mean 168.6cm, near-symmetric. Taller individuals generally jump further.'),
    ('weight_kg',               'Mean 67.1kg, slight right skew. Heavier mass can aid jump via muscle, but also increases load.'),
    ('body fat_%',              'Mean 24.0%, right-skewed. High body fat is expected to reduce jump distance.'),
    ('gripForce',               'Mean 36.9kg, near-symmetric. Strong proxy for overall muscular strength.'),
    ('sit-ups counts',          'Mean 39.8 reps, slight left skew. Core endurance linked to athletic output.'),
    ('diastolic / systolic',    'Near-normal distributions. Indirect link to cardiovascular fitness.'),
]
for col, note in notes:
    print(f"  {col:<30}: {note}")

---
## 1.8 Distribution Analysis

Histograms reveal shape, skewness, bimodality, and target distribution.

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes = axes.flatten()

for idx, col in enumerate(NUM_COLS):
    ax = axes[idx]
    series = df[col].dropna()
    skew_val = series.skew()

    is_target = col == 'broad jump_cm'
    color = 'tomato' if is_target else 'steelblue'

    sns.histplot(series, kde=True, ax=ax, color=color, edgecolor='white', alpha=0.8)
    ax.axvline(series.mean(),   color='red',    ls='--', lw=1.5, label=f'Mean={series.mean():.1f}')
    ax.axvline(series.median(), color='orange', ls='--', lw=1.5, label=f'Median={series.median():.1f}')

    shape = ('Highly Skewed' if abs(skew_val) > 1 else
             'Moderately Skewed' if abs(skew_val) > 0.5 else 'Approx Normal')
    title = f'{col}\nSkew={skew_val:.2f} | {shape}'
    if is_target:
        title = '[TARGET] ' + title
    ax.set_title(title, fontsize=8, fontweight='bold')
    ax.legend(fontsize=7)
    ax.set_xlabel('')

# Class distribution (supplementary)
ax = axes[len(NUM_COLS)]
class_counts = df['class'].value_counts().sort_index()
bars = ax.bar(class_counts.index, class_counts.values,
              color=sns.color_palette('Pastel1', 4), edgecolor='white')
ax.set_title('class column Distribution', fontsize=9, fontweight='bold')
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
            f'{val:,}', ha='center', fontsize=8)

for j in range(len(NUM_COLS)+1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distribution Analysis — All Variables (Target in Red)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("=== Distribution Key Findings ===")
print("  broad jump_cm   : Left-skewed (target) — most cluster at moderate values ~190cm.")
print("  gripForce       : Bimodal — distinct male/female peaks; strong predictor expected.")
print("  height_cm       : Bimodal — male/female subpopulation peaks.")
print("  body fat_%      : Right-skewed — high-fat outliers present; important for regression.")
print("  age             : Right-skewed — more younger people; age negatively linked to jump.")
print("  sit-ups         : Slight left skew — most achieve moderate counts.")

---
## 1.9 Outlier Detection

Boxplots identify outliers. Regression is sensitive to outliers — each decision is justified.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

outlier_records = []
for idx, col in enumerate(NUM_COLS):
    ax = axes[idx]
    series = df[col].dropna()
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out = int(((series < lo) | (series > hi)).sum())
    pct = n_out / len(series) * 100

    color = 'tomato' if col == 'broad jump_cm' else 'lightyellow'
    sns.boxplot(y=series, ax=ax, color=color, flierprops={'markersize': 3, 'alpha': 0.5})
    ax.set_title(f'{col}\n{n_out} outliers ({pct:.1f}%)', fontsize=8, fontweight='bold')
    outlier_records.append({'Column': col, 'N Outliers': n_out, '%': round(pct,2),
                            'Lower Fence': round(lo,2), 'Upper Fence': round(hi,2)})

plt.suptitle('Outlier Detection — Boxplots (Target: broad jump_cm in Red)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

display(pd.DataFrame(outlier_records))

In [ ]:
print("=== Outlier Decision Justifications (Regression Context) ===")
print()
decisions = [
    ('broad jump_cm',           'Keep', 'TARGET — 57 outliers (0.43%). Zero-jump rows retained as real low-performance data.'),
    ('height_cm',               'Cap',  '10 outliers. Extreme heights likely measurement errors. Cap to IQR fence.'),
    ('weight_kg',               'Keep', '83 outliers. Heavy individuals exist; retaining increases model generalisability.'),
    ('body fat_%',              'Keep', '77 outliers. Clinically possible; high-fat values are informative for regression.'),
    ('diastolic',               'Cap',  '54 outliers. Extreme values capped; regression more sensitive to leverage points.'),
    ('systolic',                'Cap',  '29 outliers. Same reasoning as diastolic.'),
    ('gripForce',               'Keep', '3 outliers. Near-zero values may represent genuine inability; retain.'),
    ('sit and bend forward_cm', 'Already Capped', '409 outliers resolved at validity step.'),
    ('age',                     'Keep', '0 outliers — no action.'),
    ('sit-ups counts',          'Keep', '0 outliers — no action.'),
]
for col, action, reason in decisions:
    print(f"  [{action:<15}] {col:<30}: {reason}")

print()
for col in ['height_cm', 'systolic', 'diastolic']:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df[col] = df[col].clip(Q1 - 1.5*IQR, Q3 + 1.5*IQR)
print("Capping applied to: height_cm, systolic, diastolic")
print(f"Final shape: {df.shape}")

---
## 1.10 Correlation Analysis

Identify which features have the strongest linear relationship with broad jump distance.

In [ ]:
le_gender = LabelEncoder()
le_class  = LabelEncoder()
df['gender_encoded'] = le_gender.fit_transform(df['gender'])
df['class_encoded']  = le_class.fit_transform(df['class'])

print("Gender:", dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_))))
print("Class :", dict(zip(le_class.classes_,  le_class.transform(le_class.classes_))))

In [ ]:
corr_cols = NUM_COLS + ['gender_encoded', 'class_encoded']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Pearson Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

target_corr = corr['broad jump_cm'].drop('broad jump_cm').sort_values()
print("\nCorrelation with broad jump_cm (target):")
print(target_corr.round(3).to_string())

print()
print("=== Key Correlation Insights for Regression ===")
print("  gender_encoded (r=+0.70): STRONGEST predictor — males jump significantly further.")
print("  sit-ups counts (r=+0.75): Top predictor — core endurance directly links to power.")
print("  gripForce      (r=+0.75): Top predictor — overall muscular strength enables jump.")
print("  height_cm      (r=+0.67): Taller individuals achieve longer jumps.")
print("  body fat_%     (r=-0.67): High body fat strongly reduces jump distance.")
print("  age            (r=-0.44): Older participants jump shorter distances.")
print("  weight_kg      (r=+0.48): Heavier (muscular) individuals jump further on average.")
print("  diastolic      (r=+0.10): Very weak — minimal regression signal.")

In [ ]:
# Scatter plots for top predictors vs target
top_predictors = ['gripForce', 'sit-ups counts', 'body fat_%', 'age', 'height_cm', 'weight_kg']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, col in enumerate(top_predictors):
    ax = axes[idx]
    r_val = corr.loc[col, 'broad jump_cm']
    ax.scatter(df[col], df['broad jump_cm'], alpha=0.2, s=8, color='steelblue')
    m, b = np.polyfit(df[col], df['broad jump_cm'], 1)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, m*x_line+b, 'r-', lw=2)
    ax.set_xlabel(col); ax.set_ylabel('broad jump_cm')
    ax.set_title(f'{col} vs broad jump_cm\nr = {r_val:.3f}', fontweight='bold')

plt.suptitle('Scatter Plots — Top Predictors vs Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 1.11 Final EDA Summary

In [ ]:
print("=" * 70)
print("  FINAL EDA SUMMARY — REGRESSION NOTEBOOK")
print("=" * 70)

print()
print("--- FIVE KEY INSIGHTS ---")
insights = [
    "1. Gender is the single strongest predictor of broad jump distance (r=+0.70).",
    "   Males jump on average ~40-50 cm further than females in this dataset.",
    "2. Grip force and sit-up count (both r~+0.75) are top predictors, showing",
    "   that muscular strength and endurance are the primary drivers of jump power.",
    "3. Body fat % (r=-0.67) strongly reduces jump performance — each additional",
    "   percentage point of fat is associated with a shorter jump distance.",
    "4. Age has a meaningful negative effect (r=-0.44): performance declines with age,",
    "   reinforcing the need to include age as a covariate in regression models.",
    "5. Height (r=+0.67) correlates with jump distance, but much of this is mediated",
    "   by gender — taller individuals (mostly male) have longer limb leverage.",
]
for line in insights:
    print("  " + line)

print()
print("--- FIVE POTENTIAL DATA QUALITY PROBLEMS ---")
problems = [
    "1. Impossible blood pressure values (8 rows with systolic/diastolic <= 40). Removed.",
    "2. Extreme sit-and-reach values (up to 213 cm, beyond human anatomy). Capped.",
    "3. 1 exact duplicate row in raw data. Removed.",
    "4. 10 broad jump = 0 cm entries — ambiguous (genuine failure vs. recording error).",
    "5. Strong confounding by gender: many features (height, grip) correlate with both",
    "   gender and target — regression models should include gender as a feature.",
]
for line in problems:
    print("  " + line)

print()
print("--- RECOMMENDED PREPROCESSING STEPS ---")
steps = [
    "1. Encode gender as binary (F=0, M=1) — highest-correlation predictor.",
    "2. Engineer BMI = weight / height^2 to synthesise body composition signal.",
    "3. Apply StandardScaler to all numeric features (fit on train set only).",
    "4. Use KFold (k=5) cross-validation with R^2 and MAE as evaluation metrics.",
    "5. Investigate gender-stratified models since jump distance distributions differ.",
]
for line in steps:
    print("  " + line)

print()
print(f"Dataset shape after full EDA cleaning: {df.shape}")

---
## 2. Feature Engineering

In [ ]:
# BMI feature
df['BMI'] = df['weight_kg'] / ((df['height_cm'] / 100) ** 2)
print(f"BMI created: mean={df['BMI'].mean():.2f}, std={df['BMI'].std():.2f}")

# Target and features — exclude broad jump from predictors
FEATURES = ['age','height_cm','weight_kg','body fat_%','diastolic','systolic',
            'gripForce','sit and bend forward_cm','sit-ups counts',
            'class_encoded','BMI','gender_encoded']
TARGET = 'broad jump_cm'

X = df[FEATURES]
y = df[TARGET]
print(f"\nFeature matrix : {X.shape}")
print(f"Target vector  : {y.shape}")
print(f"\nTarget stats:")
print(f"  Mean={y.mean():.2f}  Median={y.median():.2f}  Std={y.std():.2f}  Min={y.min():.1f}  Max={y.max():.1f}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Train : {X_train_s.shape}")
print(f"Test  : {X_test_s.shape}")
print("StandardScaler fitted on train only — no data leakage.")

---
## 3. Model Building & Evaluation

